# Example 3.3 / 1: Entities and expressions

Companion notebook for **3.3 Displaying Results** in the AMPL lab Documentation, section "Entities and expressions". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Entities and expressions

The model below is solved once. `to_dict()` returns the indexed variable values, while
`get_data()` evaluates a new indexed expression without adding an entity to the model.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;
param profit {PROD};
param market {PROD} >= 0;
var Make {p in PROD} >= 0, <= market[p];
maximize Total_Profit: sum {p in PROD} profit[p] * Make[p];

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils"]
ampl.param["profit"] = {"bands": 25, "coils": 30}
ampl.param["market"] = {"bands": 6000, "coils": 4000}

In [5]:
# run.py
ampl.solve(solver="highs")
print(ampl.var["Make"].to_dict())
revenue = ampl.get_data("{p in PROD} profit[p] * Make[p]")
print(revenue.to_dict())

HiGHS 1.14.0: optimal solution; objective 270000
0 simplex iterations
0 barrier iterations
{'bands': 6000, 'coils': 4000}
{'bands': 150000, 'coils': 120000}


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set PROD := bands coils;
param: profit market :=
  bands 25 6000
  coils 30 4000;

Writing data.dat


In [7]:
%%ampl_eval
reset;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
solve;
display Make;
display {p in PROD} profit[p] * Make[p];

HiGHS 1.14.0: optimal solution; objective 270000
0 simplex iterations
0 barrier iterations
Make [*] :=
bands  6000
coils  4000
;

profit[p]*Make[p] [*] :=
bands  150000
coils  120000
;

